## **LangChain** - _Retrieval-Augmented Generation_

Llegamos a la pieza central del desarrollo moderno con **LLMs**: **Retrieval-Augmented Generation** - _Generación Aumentada por Recuperación_. 

Los **LLMs** enfrentan un desafío importante: las "alucinaciones" y la falta de acceso a información privada o actualizada en tiempo real. 

La **Generación Aumentada por Recuperación** (_RAG_) es una técnica que mejora las respuestas de un modelo de lenguaje conectándolo a una base de conocimientos externa. En lugar de depender únicamente de los datos con los que fue entrenado, el modelo primero "busca" información relevante en documentos específicos antes de generar una respuesta.

El flujo de trabajo típico de un sistema **RAG** se divide en varias etapas clave:

1. **Carga de datos:** Se utilizan herramientas como **Loaders** para ingerir documentos.
2. **División de texto (Chunking):** El texto extenso se corta en fragmentos más pequeños para que sea manejable.
3. **Embeddings:** Los fragmentos se convierten en vectores matemáticos y se almacenan en una base de datos vectorial.
4. **Recuperación y Generación:** Cuando un usuario hace una pregunta, el sistema busca los fragmentos más relevantes y se los pasa al **LLM** para que construya una respuesta precisa.

En este notebook utilizaremos todas las herramientas aprendidas a la vez que repasaremos conceptos importantes de cada herramienta:
1. Loaders
2. Text Splitters
3. Embeddings
4. ChromaDB
5. LLMs
6. Parsers
7. LCEL

In [1]:
from dotenv import load_dotenv

# Variables de entorno
load_dotenv()

MODEL = "gemini-3.1-flash-lite"
TEMPERATURE = 0

### 01. **Loaders**

Los **LLMs** no conocen tu información privada. El primer paso del **RAG** es importar datos desde una fuente (PDFs, bases de datos, webs, etc.) hacia nuestro entorno. En este ejemplo, cargaremos un archivo **.txt**.

In [2]:
from langchain_community.document_loaders import TextLoader

# Cargar el documento
loader = TextLoader(file_path = "../Data/harry_potter_books/01.harry_potter_piedra_filosofal.txt", encoding = "utf-8")
documentos = loader.load()

print(f"Se ha cargado {len(documentos)} documento(s).")
print("Contenido inicial:", documentos[0].page_content, "...")

C:\Users\danie\AppData\Local\Temp\ipykernel_3112\121816831.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader
c:\Users\danie\OneDrive\Documentos\GitHub\DAIXXRT\LangChain - Boost Academy\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Se ha cargado 1 documento(s).
Contenido inicial: J.K. ROWLING
Harry Potter
y
la piedra filosofal
Harry Potter se ha quedado huérfano y vive en casa de sus abominables tíos y del
insoportable primo Dudley. Harry se siente muy triste y solo, hasta que un buen
día recibe una carta que cambiará su vida para siempre. En ella le comunican que
ha sido aceptado como alumno en el colegio interno Hogwarts de magia y
hechicería. A partir de ese momento, la suerte de Harry da un vuelco
espectacular. En esa escuela tan especial aprenderá encantamientos, trucos
fabulosos y tácticas de defensa contra las malas artes. Se convertirá en el
campeón escolar de quidditch, especie de fútbol aéreo que se juega montado
sobre escobas, y se hará un puñado de buenos amigos... aunque también algunos
temibles enemigos. Pero sobre todo, conocerá los secretos que le permitirán
cumplir con su destino. Pues, aunque no lo parezca a primera vista, Harry no es
un chico común y corriente. ¡Es un mago!
Título original: Har

In [3]:
len(documentos[0].page_content)

454608

### 02. **Text Splitters**

Los **LLMs** tienen un límite de contexto (cuántas palabras pueden leer a la vez). No podemos pasar un libro entero. Debemos dividir el texto en fragmentos más pequeños o **chunks**.

Un parámetro importante es el **overlap** (_superposición_), que evita que una idea importante se corte a la mitad de una frase.

In [4]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Dividimos en trozos de 1000 caracteres con una superposición de 150
text_splitter = RecursiveCharacterTextSplitter(chunk_size = 1000,
                                               chunk_overlap = 150,
                                               separators = ["\n\n", "\n", " ", ""])

fragmentos = text_splitter.split_documents(documents = documentos)

fragmentos

[Document(metadata={'source': '../Data/harry_potter_books/01.harry_potter_piedra_filosofal.txt'}, page_content='J.K. ROWLING\nHarry Potter\ny\nla piedra filosofal\nHarry Potter se ha quedado huérfano y vive en casa de sus abominables tíos y del\ninsoportable primo Dudley. Harry se siente muy triste y solo, hasta que un buen\ndía recibe una carta que cambiará su vida para siempre. En ella le comunican que\nha sido aceptado como alumno en el colegio interno Hogwarts de magia y\nhechicería. A partir de ese momento, la suerte de Harry da un vuelco\nespectacular. En esa escuela tan especial aprenderá encantamientos, trucos\nfabulosos y tácticas de defensa contra las malas artes. Se convertirá en el\ncampeón escolar de quidditch, especie de fútbol aéreo que se juega montado\nsobre escobas, y se hará un puñado de buenos amigos... aunque también algunos\ntemibles enemigos. Pero sobre todo, conocerá los secretos que le permitirán\ncumplir con su destino. Pues, aunque no lo parezca a primera vis

In [10]:
len(fragmentos)

532

### 03. Vectorizar y Almacenar (**Embeddings** & **ChromaDB**)

Para que los modelos entiendan el texto, debemos convertir cada fragmento en un **Embedding**. Los textos con significados similares tendrán vectores cercanos en un espacio multidimensional.

Guardamos en una instancia de **ChromaDB** los **embeddings** generados.

Para el modelo de **Embeddings** usaremos **BAAI/bge-m3** (gratuito en _HuggingFace_):

- Es un modelo de lenguaje de código abierto desarrollado por la **Beijing Academy of Artificial Intelligence**.

- Su nombre proviene de tres características clave: **Multilingüe**, **Multifuncional** y **Multigranular**.

    - **Multilingüe**: Soporta más de 100 idiomas (incluyendo español) y permite búsquedas multilingües (por ejemplo, buscar texto en español a partir de una consulta en inglés).

    - **Multifuncional**: Combina tres métodos de búsqueda:
        - **Recuperación densa** (vectores semánticos).
        - **Recuperación dispersa** (palabras clave tradicionales).
        - **Recuperación multivectorial** (detalles más profundos).

    - **Multigranular**: Puede procesar desde oraciones muy cortas hasta documentos muy extensos, admitiendo hasta 8192 tokens de contexto.


In [ ]:
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings

# Definimos el modelo de embeddings
modelo_embeddings = HuggingFaceEmbeddings(model = "BAAI/bge-m3")

# Conectamos e inicializamos nuestra colección en ChromaDB
# Chroma creará la carpeta automáticamente si no existe para guardar los vectores
vectorstore = Chroma(embedding_function = modelo_embeddings,
                     collection_name = "rag_ejemplo",
                     persist_directory = "../rag_ejemplo")

# Guardamos los fragmentos (chunks) generados en el punto anterior en la base de datos
vectorstore.add_documents(documents = fragmentos)

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 10494.82it/s]


['37a777a5-f952-492d-9ada-1034e64a4aa6',
 '81bc212b-ff5c-42f8-a8cf-b24c5241d509',
 'd11c1f1a-81ed-418f-887f-d3301d9481ab',
 'a0203ac8-4703-4a21-b54d-6cce07ec0114',
 '5e661da8-dbe6-47a2-bc02-f177115c1cb4',
 'eee79337-6d59-4525-9b05-ff02c7120d12',
 'f61277d6-f99e-41e6-8c7e-456fd0d434f9',
 'ae12d9a8-f6b6-45c8-aa80-b7a71c865848',
 '47802b8e-61d9-4c21-91e2-dea19135a60d',
 'fd888b24-9577-403b-a731-16171e988bdb',
 '414422cc-516c-4ede-8285-2bcab8149341',
 '538f270b-0859-43d2-a327-9837298ebc1c',
 '6bb8e7eb-e086-4b38-8bc3-89320203b98e',
 'a2f1eff7-19c2-47fa-abef-de541045f36f',
 'd1d8439a-09e6-4605-ab41-3f79d90cd4a1',
 '9acc9cb8-f26d-4fbb-a847-650a77cae196',
 '89ad040d-be2f-42e0-8cb8-b00e9f93e2b1',
 'f7c5ad06-8d2d-4b84-823f-ef1266d39cfb',
 '5fab9070-c5d6-452b-a152-05b4263a8d5a',
 'f49ba3b0-cf6c-4ee0-acb6-695803ef5f69',
 'e98cb2e9-aee5-4334-8214-d935eed8498d',
 '201e37cd-3ab7-4ff6-b536-f34c00af716b',
 'ab3a7aaf-5052-4796-a8fb-c74de319ead8',
 'bf2acedb-19ff-43f7-b919-39c2471deda7',
 'ccec9254-2275-

### 04. Retrieve

Cuando el usuario hace una pregunta, primero convertimos esa pregunta en un **embedding** y buscamos en nuestra base de datos cuáles son los fragmentos de texto más cercanos matemáticamente a la pregunta.

In [18]:
# Convertimos el vectorstore en un motor de búsqueda
retriever = vectorstore.as_retriever(search_kwargs = {"k" : 2})

pregunta_usuario = "que es la piedra filosofal?"

# Realizamos la búsqueda semántica
contexto_recuperado = retriever.invoke(input = pregunta_usuario)

print("Fragmentos recuperados para responder a la pregunta:")
for i, doc in enumerate(contexto_recuperado):
    print(f"\nResultado {i+1}:\n{doc.page_content}")

Fragmentos recuperados para responder a la pregunta:

Resultado 1:
de él.
—Nicolás Flamel —susurró con tono teatral— es el único descubridor conocido de
la Piedra Filosofal.
Aquello no tuvo el efecto que ella esperaba.
—¿La qué? —dijeron Harry y Ron.
—¡Oh, no lo entiendo! ¿No sabéis leer? Mirad, leed aquí. Empujó el libro hacia
ellos, y Harry y Ron leyeron:
El antiguo estudio de la alquimia está relacionado con el descubrimiento de la
Piedra Filosofal, una sustancia legendaria que tiene poderes asombrosos. La
piedra puede transformar cualquier metal en oro puro. También produce el
Elixir de la Vida, que hace inmortal al que lo bebe.
Se ha hablado mucho de la Piedra Filosofal a través de los siglos, pero la
única Piedra que existe actualmente pertenece al señor Nicolás Flamel, el
notable alquimista y amante de la ópera. El señor Flamel, que cumplió
seiscientos sesenta y cinco años el año pasado, lleva una vida tranquila en
Devon con su esposa Perenela (de seiscientos cincuenta y ocho añ

### 04. Ensamblando el Sistema **RAG**

Vamos a unir todo lo que hemos aprendido:
1. El usuario hace una pregunta.
2. El `retriever` busca en **ChromaDB** el contexto necesario.
3. Inyectamos la pregunta y el contexto en un `ChatPromptTemplate`.
4. El modelo (`ChatGoogleGenerativeAI`) lee el contexto y genera una respuesta.
5. El `StrOutputParser` nos entrega el texto limpio.

In [17]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# Instanciamos el modelo y el parser
llm = ChatGoogleGenerativeAI(model = MODEL, temperature = TEMPERATURE)
parser = StrOutputParser()
retriever = vectorstore.as_retriever(search_kwargs = {"k": 2})

# Creamos el Prompt RAG
template_rag = """
Responde a la pregunta del usuario utilizando ÚNICAMENTE el siguiente contexto. 
Si la respuesta no está en el contexto, di "No tengo información sobre eso".

Contexto recuperado:
{contexto}

Pregunta del usuario:
{pregunta}
"""
prompt_rag = ChatPromptTemplate.from_template(template = template_rag)

# Función auxiliar para unir los textos de los documentos en un solo gran string
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# Cadena RAG (LCEL)
# Usamos un diccionario dinámico para poblar las variables del prompt
cadena_rag = ({"contexto": retriever | format_docs, "pregunta": RunnablePassthrough()}
              | prompt_rag
              | llm
              | parser)

pregunta_final = "que es la piedra filosofal?"

respuesta_final = cadena_rag.invoke(input = pregunta_final)

print("Respuesta de Gemini basada en nuestros documentos locales:")
print(respuesta_final)

Respuesta de Gemini basada en nuestros documentos locales:
La Piedra Filosofal es una sustancia legendaria que tiene poderes asombrosos: puede transformar cualquier metal en oro puro y produce el Elixir de la Vida, que hace inmortal al que lo bebe.
